### News

Placeholder

### Installation

In [1]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29 peft trl triton
    !pip install --no-deps cut_cross_entropy unsloth_zoo
    !pip install sentencepiece datasets huggingface_hub hf_transfer
    !pip install protobuf==3.20.3 # required
    !pip install --no-deps transformers-cfg
    # !pip install --no-deps unsloth
    # !git clone https://github.com/jedt/unsloth.git

In [2]:
%%capture
!pip install --no-deps bitsandbytes accelerate xformers==0.0.29 peft trl triton

In [3]:
%%capture
!pip install "git+https://github.com/unslothai/unsloth@main#egg=unsloth[cu121onlytorch251]"

In [4]:
%%capture
!pip install --no-deps torchvision==0.20.1
!pip install --no-deps torchaudio==2.5.1

In [5]:
%%capture
!pip install transformers==4.47.1 --ignore-installed

In [6]:
%%capture
!pip uninstall unsloth -y
# use fork for now
!git clone https://github.com/jedt/unsloth.git

You may need to restart the session before running the cell below

In [1]:
%%capture
import os
import sys
sys.path.append(os.path.abspath("/content/unsloth"))
!cd /content/unsloth && pip install -e .

<img src="https://cdn-lfs.hf.co/datasets/huggingface/documentation-images/48d216f11e3d8fe183d36c84f1380b54dd18608be60224bd3785007be8d567a7?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27function-callin.png%3B+filename%3D%22function-callin.png%22%3B&response-content-type=image%2Fpng&Expires=1741999405&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc0MTk5OTQwNX19LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy5oZi5jby9kYXRhc2V0cy9odWdnaW5nZmFjZS9kb2N1bWVudGF0aW9uLWltYWdlcy80OGQyMTZmMTFlM2Q4ZmUxODNkMzZjODRmMTM4MGI1NGRkMTg2MDhiZTYwMjI0YmQzNzg1MDA3YmU4ZDU2N2E3P3Jlc3BvbnNlLWNvbnRlbnQtZGlzcG9zaXRpb249KiZyZXNwb25zZS1jb250ZW50LXR5cGU9KiJ9XX0_&Signature=mmBIprXrDD88zZYSN8XglAd1wdmb4NZOZrDvXU7BSpzBn1tJE3MiJ42fruvFmB0yQ6dOsAL8wzn3Xf6MZdJCvOwxtkaPUOe5pp%7Ej7mO1E23FIDNEX0NJAM%7ExUTCCoRBYo3ASnlQEe8tOYwbfpES2CJw-xHHnZK7WTsgGqJBtyyQ20fuT7M1ahPmeS8MLjLJlH9BvQeREP%7EkgfkWj6Cw6Zxzd78gBXs-fVlPlFy2cHD6zKkof-p4tCMSvELnXnT89mIXlpUKcN%7EgmoN38yr90fXU4OYpp9mreeSKQyHtA4D4nfqXh5c4B5b%7EIzssOQYltofpApfVYIHPsKKeuK6Jgcw__&Key-Pair-Id=K3RPWS32NSSJCE">

Image from [How Function Calling Works](https://huggingface.co/docs/hugs/en/guides/function-calling)

### Tool Calling
By definition, Tool Calling refers to the capability of models to interact with external tools. As you may already know, LLMs are excellent at generating text in the manner of question-and-answer pairs, chat-like conversation messages, and others. When provided useful context, LLMs are good at responding to prompts that can be used as actionable command parameters that another system would handle. This could be a search engine, calculators, company APIs, spreadsheets, or SQL queries. These systems when interacted with are usually done in a single command line or if more complex a runnable scripting language such as Bash or Python. The painful part is how to sequence these commands and assign the correct parameter names and values. What if we would just prompt the LLM to do these for us? For example. "Schedule an appointment for Alice with Bob Monday, 10:00am Eastern time"

### Unsloth

For this notebook, we'll use a smaller Qwen2.5 model 1.5B. We will guide you on how to prompt a model to respond in JSON format so we can then parse the results and pass those arguments to a Python script. The intuition for using a small model is that we do not want the model to use its pretraining data for the responses when calculating the vector sum. Smaller models have less stored knowledge and it would be possible that our prompt is not on the training data.

Our intention is to provide a simple framework for integrating tool calling into your fine-tuning workflow for unsloth. Let us know how we can help you further.

In [2]:
from unsloth import FastQwen2Model
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 2x faster
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Mistral-Small-Instruct-2409",     # Mistral 22b 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!

    "unsloth/Llama-3.2-1B-bnb-4bit",           # NEW! Llama 3.2 models
    "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
] # More models at https://huggingface.co/unsloth

qwen_models = [
    "unsloth/Qwen2.5-Coder-32B-Instruct",      # Qwen 2.5 Coder 2x faster
    "unsloth/Qwen2.5-Coder-7B",
    "unsloth/Qwen2.5-14B-Instruct",            # 14B fits in a 16GB card
    "unsloth/Qwen2.5-7B",
    "unsloth/Qwen2.5-72B-Instruct",            # 72B fits in a 48GB card
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastQwen2Model.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-1.5B-Instruct",
    max_seq_length = None,
    dtype = None,
    load_in_4bit = False,
    fix_tokenizer = False
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.3.10: Fast Qwen2 patching. Transformers: 4.47.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


config.json:   0%|          | 0.00/765 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.51k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

### Tool Definitions
This is a list of all the functions that we would provide to the model. The standard format is from OpenAI's [Function Calling Definition](https://platform.openai.com/docs/guides/function-calling?api-mode=chat#defining-functions)
It is highly possible that the model trained for tool calling was in the OpenAI standard format.

Below is an example of two function definitions. The function definitions of `get_vector_sum` and `get_dot_product` will then be added to the prompt as a context for our prompt:

> Find the sum of a = [1, -1, 2] and b = [3, 0, -4].

We can just provide the correct one: `get_vector_sum` but to experiment if the model can identify the correct function to call, we will provide both.

In [4]:
def get_tool_definition_list():
    return [
        {
            "type": "function",
            "function": {
                "name": "get_vector_sum",
                "description": "Get the sum of two vectors",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "a": {"type": "list", "description": "First vector"},
                        "b": {"type": "list", "description": "Second vector"}
                    },
                    "required": ["a", "b"]
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "get_dot_product",
                "description": "Get the dot product of two vectors",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "a": {"type": "list", "description": "First vector"},
                        "b": {"type": "list", "description": "Second vector"}
                    },
                    "required": ["a", "b"]
                }
            }
        },

    ]

Below is the user prompt declaration

In [5]:
user_query = {
    "role": "user",
    "content": "Find the sum of a = [1, -1, 2] and b = [3, 0, -4]."
}

### Python Code
Below is the actual code for the function, you may notice that it has Python docstrings. This is because `apply_chat_template()` can accept and translate functions into OpenAI function definitions that are PEP 257 compliant.

In [6]:
def get_vector_sum(a: list[float], b: list[float]) -> list[float]:
    """
    Performs element-wise addition of two numerical vectors.

    Both vectors must be of the same length and contain numerical values.

    Args:
        a: First vector containing numerical values
        b: Second vector containing numerical values

    Returns:
        Resulting vector where each element is the sum of corresponding elements in a and b

    Raises:
        ValueError: If vectors have different lengths

    Example:
        >>> get_vector_sum([1, 2], [3, 4])
        [4, 6]
    """
    if len(a) != len(b):
        raise ValueError("Vectors must be of the same length")

    return [x + y for x, y in zip(a, b)]

Now let's prompt the model to provide us the arguments in JSON format may notice that we passed the actual function `get_vector_sum()` to the `tool` parameter and not the `get_tool_definition_list()`, You may try to change it from `tools=[get_vector_sum],` to `tools=[get_tool_definition_list()` to see if it works with the function definitions.

In [9]:
messages = []

messages.append(user_query)

input_ids = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    add_special_tokens=False,
    padding=True,
    tools=[get_vector_sum],
    return_tensors = "pt",
).to("cuda")

print(tokenizer.decode(input_ids[0]))

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "get_vector_sum", "description": "Performs element-wise addition of two numerical vectors.\n\nBoth vectors must be of the same length and contain numerical values.", "parameters": {"type": "object", "properties": {"a": {"type": "array", "items": {"type": "number"}, "description": "First vector containing numerical values"}, "b": {"type": "array", "items": {"type": "number"}, "description": "Second vector containing numerical values"}}, "required": ["a", "b"]}, "return": {"type": "array", "items": {"type": "number"}, "description": "Resulting vector where each element is the sum of corresponding elements in a and b"}}}
</tools>

For each function call, return a json object with function name and argume

Below is where we call the unsloth function `generate_with_grammar()`. This function uses Grammar-Constrained Decoding. Meaning it will only respond in JSON. It uses a library fork of [transformers-CFG](https://github.com/Saibo-creator/transformers-CFG) by [Saibo-creator](https://github.com/Saibo-creator) the output is very similar to the llama-cpp-python output. We decided to use this library so it is easier for you to migrate to `llama-cpp-python` later during production.

If successful, the model should output a single valid JSON response with the following result:
```
[
    {
        "name": "get_vector_sum",
        "arguments": {
            "a": [1, -1, 2],
            "b": [3, 0, -4]
        }
    }
]
```

In [10]:
from unsloth.models.llama import generate_with_grammar
output = generate_with_grammar(
    model=model,
    input_ids=input_ids
)

generated_tokens = output[:, input_ids.shape[1]:]

decoded_output = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)

for i, message in enumerate(decoded_output):
    print(f"{message}")

tokenizer_config.json:   0%|          | 0.00/7.51k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


[
    {
        "name": "get_vector_sum",
        "arguments": {
            "a": [1, -1, 2],
            "b": [3, 0, -4]
        }
    }
]


From here we can now parse the arguments provided to use by the model

In [11]:
import json
content = json.loads(decoded_output[0])
arguments = content[0]['arguments']
vector_a = arguments['a']
vector_b = arguments['b']
print(f"args a: {vector_a}, b: {vector_b}")

args a: [1, -1, 2], b: [3, 0, -4]


Here we actually call `get_vector_sum()` and capture the result

In [12]:
result = get_vector_sum(vector_a, vector_b)
print(f"result: {result}")

result: [4, -1, -2]


Below is the final prompt to the model in the form of a chat message. To ensure that the model responds with the actual answer  prompted the model's answer with the following:

> You are a super helpful AI assistant. You are asked to answer a question based on the following context information.
>
> Question:
>
> Answer:

Then we set `continue_final_message=True` for the tokenizer

In [29]:
import random
import string

def generate_alphanumeric():
    characters = string.ascii_letters + string.digits
    result = ''.join(random.choice(characters) for _ in range(9))
    return result

messages = []

original_prompt = user_query['content']
prompt_with_context = f"""You are a super helpful AI assistant.
You are asked to answer a question based on the following context information.\n
Question:\n
{original_prompt}"""

user_query = {
    "role": "user",
    "content": prompt_with_context
}
messages.append(user_query)
tool_call_id = generate_alphanumeric()
tool_calls = [{
            "id": tool_call_id,
            "type": "function",
            "function": {
                "name": "get_vector_sum",
                "arguments": arguments
            }
        }]


messages.append({
    "role": "assistant",
    "tool_calls": tool_calls
})
messages.append({
    "role": "tool",
    "name": "get_vector_sum",
    "content": result
})

messages.append({
    "role": "assistant",
    "content": "Answer:\n"
})

tool_prompt = tokenizer.apply_chat_template(
    messages,
    continue_final_message=True,
    add_special_tokens=True,
    return_tensors="pt",
    return_dict=True,
)
tool_prompt = tool_prompt.to(model.device)

print(tokenizer.decode(tool_prompt['input_ids'][0]))

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
You are a super helpful AI assistant.
You are asked to answer a question based on the following context information.

Question:

You are a super helpful AI assistant.
You are asked to answer a question based on the following context information.

Question: Find the sum of a = [1, -1, 2] and b = [3, 0, -4].<|im_end|>
<|im_start|>assistant
<tool_call>
{"name": "get_vector_sum", "arguments": {"a": [1, -1, 2], "b": [3, 0, -4]}}
</tool_call><|im_end|>
<|im_start|>user
<tool_response>
[4, -1, -2]
</tool_response><|im_end|>
<|im_start|>assistant
Answer:



In [32]:
out = model.generate(**tool_prompt, max_new_tokens=128)
generated_text = out[0, tool_prompt['input_ids'].shape[1]:]

print(tokenizer.decode(generated_text, skip_special_tokens=True))

The sum of the vectors a = [1, -1, 2] and b = [3, 0, -4] is [4, -1, -2].
